## large target array synthesis for base editing palindromes

Our goal is to make one construct with 9 guides expressed off of individual U6s, editing
a large number of target sequences that are expressed in the 3' UTR of a gene. 

In [1]:
targets = ["TCATACGGGCGCCCGTATGAGGG",
          "CGCATAAGCCGGCTTATGCGTGG",
          "AGTATAGCGCGCGCTATACTGGG",
          "GTATAGCGGATCCGCTATACCGG",
          "GCGATAGGACGTCCTATCGCTGG",
          "TCGGTTACGCGCGTAACCGACGG",
          "GCGGTACTTCGAAGTACCGCCGG",
          "CCGATAGTGTACACTATCGGCGG",
          "TAGATAGTCGCGACTATCTAAGG"]

##### Base vector

pLarry - https://benchling.com/s/seq-uWz0fBZh22whcaCuGtr3

- We'll cut it with SalI and XbaI to add the insert for the targets
- We'll cut with SpeI for the guides. This means they could go in either direction; we'll have to search after the fact



In [88]:
# The minimal U6 sequence:
minimal_u6 = "GAGGGCCTATTTCCCATGATTCCTTCATATTTGCATATACGATAGCTTACCGTAACTTGAAAGTATTTCGATTTCTTGGCTTTATATATCTTGTGGAAAGGACGAAACACCG"
print(len(minimal_u6))

# our guide scaffold plus terminator
guide_scaffold = "GTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGCACCGAGTCGGTGCTTTTTTG"
print(len(guide_scaffold))

print(len(minimal_u6) + len(guide_scaffold))


112
83
195


#### Some simple function to create padded sequences

In [1]:
import random
def random_sequence(length):
    return("".join(random.choices(['A','C','G','T'],k=length)))

def gc(seq):
    return((sum([1 if x == 'C' or x == 'G' else 0 for x in seq]))/len(seq))

def comp(x):
    if x == 'A':
        return('T')
    elif x == 'C':
        return('G')
    elif x == 'G':
        return('C')
    elif x == 'T':
        return('A')
    else:
        return('N')
    
def revcomp(st):
    xxx = [comp(x) for x in st]
    xxx.reverse()
    return("".join(xxx))

def screen_gc(minval,maxval,length):
    score = -1
    newseq = ""
    while score < minval or score > maxval:
        newseq = random_sequence(length)
        score = gc(newseq)
    return(score,newseq)

#for i in range(0,10):
#    print(screen_gc(0.4,0.6,20))

In [5]:
for i in range(0,10):
    print(screen_gc(0.4,0.6,60))

(0.43333333333333335, 'CTGGAAAGAGTCGTCAGACAGGGGAGATAAATGCGACAATTACACGATTTAATGCTGAAC')
(0.43333333333333335, 'CTTAAGCTGACATTTATGGTGGGAATGACGTCTCCGCATTAAGAACATGAGCGCATTTAG')
(0.45, 'GCGATTTCATAACGCGGATGTGGACCAAAATACATTGTTTAAGCATAACGGCACCCTGCA')
(0.43333333333333335, 'TCGTGACTCTTGTACGAATATGATATCCGTCCTAGACATTAGCTTCGAAGGTATGGACTC')
(0.5333333333333333, 'TACACGCCTACTGGCCGTATAGAAGACCTAGTAACGTAGTGGCCTAGTGTGAGGCTCGAC')
(0.4666666666666667, 'ATTTTATGAGAGGGCCCGTGGACTCAATGCCGATCCGTATTCCACATAAGGAGACATATG')
(0.4666666666666667, 'GCATATCATATGGGGAACCCGTTTGATGCCGAACCTGATGTATGTCGCTAACGAGATTAC')
(0.5166666666666667, 'ATGTGGGCGAGGATCTGGACCAAACCGGGATAGTTGTCTCCAGAGTTCTCGTAATGTACG')
(0.5, 'ATGACCACTGGGCCTTTAAACACAAATTCAACAAACGCCGCGTACACGTTCCGCCACCAT')
(0.5166666666666667, 'TGCCATAGATTAATTAGCGTTGGTTGCTGAGTGACCCAAGCTTCTACATGCGCTCGCCGG')


In [55]:
#primer1 = screen_gc(0.45,0.55,20)
#primer2 = screen_gc(0.45,0.55,20)
primer1 = "GTAGCAGATGTGGCCTTATG"
primer2 = "GCCAGATACCCGAGACTAC"


print(primer1)
print(primer2)

GTAGCAGATGTGGCCTTATG
GCCAGATACCCGAGACTAC


In [30]:
bsmbi_sequence = "CGTCTC"
bsmbi_rc_sequence = revcomp(bsmbi_sequence)
print(bsmbi_rc_sequence)

GAGACG


In [32]:
def get_rc_pam(target):
    return(revcomp(target[20:23]))
print(targets[1])
print(get_rc_pam(targets[1]))

CGCATAAGCCGGCTTATGCGTGG
CCA


In [66]:
# https://pubs.acs.org/doi/10.1021/acssynbio.8b00333
overhangs = ['TGCC', 'GCAA', 'ACTA', 'TTAC', 'TGTG', 'GAGC', 'AGGA', 'ATTC', 'CGAA', 'ATAG', 'AAGG', 'AACT', 'AAAA', 'ACCG']
overhangs2 = ['AGTG', 'CAGG', 'ACTC', 'AAAA', 'AGAC', 'CGAA', 'ATAG', 'AACC', 
              'TACA', 'TAGA', 'ATGC', 'GATA', 'CTCC', 'GTAA', 'CTGA', 'ACAA', 'AGGA', 'ATTA', 'ACCG', 'GCGA']

### Our design pattern

---primer 1 --> RE1 --- target1 --- target2 --- target1 --- target2 --- target1 --- target2 --- RE2 <-- primer2 ---

In [57]:
def design_target_set(target1,target2,re1,re2,primer1,primer2):
    sequence = primer1 + bsmbi_sequence + "T" + re1
    for i in range(0,3):
        sequence += screen_gc(0.32,0.67,3)[1] + "A" + get_rc_pam(target1) + target1 + "T" + screen_gc(0.32,0.67,3)[1]
        sequence += "A" + get_rc_pam(target2) + target2 + "T"
    sequence += screen_gc(0.32,0.67,3)[1] + re2 + "A" + bsmbi_rc_sequence + revcomp(primer2)
    return(sequence)

st1 = design_target_set(targets[0],targets[1],overhangs[0],overhangs[1],primer1,primer2)
print(st1)
print(len(st1))

GTAGCAGATGTGGCCTTATGCGTCTCTTGCCGGTACCCTCATACGGGCGCCCGTATGAGGGTAGTACCACGCATAAGCCGGCTTATGCGTGGTAACACCCTCATACGGGCGCCCGTATGAGGGTCACACCACGCATAAGCCGGCTTATGCGTGGTACCACCCTCATACGGGCGCCCGTATGAGGGTTAGACCACGCATAAGCCGGCTTATGCGTGGTCATGCAAAGAGACGGTAGTCTCGGGTATCTGGC
250


In [164]:
outputfile = open("/Users/aaronmck/Desktop/target_set.fasta","w")
for i in range(0,4):
    targetseq = design_target_set(targets[i*2],targets[(i*2)+1],overhangs[i],overhangs[i+1],primer1,primer2)
    outputfile.write(">AM_24set_" + str(i) + "\n" + targetseq + "\n")


In [63]:
primer1s = [screen_gc(0.45,0.55,20) for x in range(8)]
primer2s = [screen_gc(0.50,0.60,19) for x in range(8)]
print(primer1s)
print(primer2s)

[(0.5, 'CTGGTATTTGGGGGTCACTA'), (0.55, 'GCTGGATATTCCCTTGGGTC'), (0.55, 'CCGTTATGGGCTAACCTGAG'), (0.45, 'GCGGTATTCGTGATCTATAG'), (0.55, 'ACGTACCGTTCTCACATCGC'), (0.55, 'TCCATCTCTGGTGCGCAATC'), (0.45, 'TGTAGCGCTTGGTAGACATA'), (0.5, 'TGGTAGACAACCACCTGAGT')]
[(0.5263157894736842, 'AGTGACGGGAGATTGCATG'), (0.5789473684210527, 'GCGCGACCAAATGTATGGC'), (0.5789473684210527, 'TTGCGGCGTGTCTGCTCAA'), (0.5263157894736842, 'AAACACACCGGCACTGCAA'), (0.5789473684210527, 'GTGTACTGCGAAGTTGCCC'), (0.5263157894736842, 'TGCTCTTACGGGACATCCA'), (0.5263157894736842, 'GGTTGCGTCAGAGTAGTCT'), (0.5789473684210527, 'CACAGTCCCTGGCATTCCT')]


In [125]:
primer1s_accepted = ['CTGGTATTTGGGGGTCACTA','GCTGGATATTCCCTTGGGTC','CCGTTATGGGCTAACCTGAG','TGTAGCGCTTGGTAGACATA','GACAACCACCTGAGT']
primer2s_accepted = ['AGTGACGGGAGATTGCATG','TGCTCTTACGGGACATCCA','GGTTGCGTCAGAGTAGTCT','CACAGTCCCTAGCATTCCT','GGAGACCATATGGC']


In [165]:
for i in range(0,4):
    targetseq = design_target_set(targets[i*2],targets[(i*2)+1],overhangs[i],overhangs[i+1],primer1s_accepted[0],primer2s_accepted[0])
    outputfile.write(">AM_48set_" + str(i) + "\n" + targetseq + "\n")
for i in range(0,4):
    targetseq = design_target_set(targets[i*2],targets[(i*2)+1],overhangs[i+5],overhangs[i+6],primer1s_accepted[0],primer2s_accepted[0])
    outputfile.write(">AM_48set_" + str(i+5) + "\n" + targetseq + "\n")



In [166]:
for i in range(0,4):
    targetseq = design_target_set(targets[i*2],targets[(i*2)+1],overhangs2[i],overhangs2[i+1],primer1s_accepted[1],primer2s_accepted[1])
    outputfile.write(">AM_96set_" + str(i) + "\n" + targetseq + "\n")
for i in range(0,4):
    targetseq = design_target_set(targets[i*2],targets[(i*2)+1],overhangs2[i+5],overhangs2[i+6],primer1s_accepted[1],primer2s_accepted[1])
    outputfile.write(">AM_96set_" + str(i+5) + "\n" + targetseq + "\n")
for i in range(0,4):
    targetseq = design_target_set(targets[i*2],targets[(i*2)+1],overhangs2[i+10],overhangs2[i+11],primer1s_accepted[1],primer2s_accepted[1])
    outputfile.write(">AM_96set_" + str(i) + "\n" + targetseq + "\n")
for i in range(0,4):
    targetseq = design_target_set(targets[i*2],targets[(i*2)+1],overhangs2[i+15],overhangs2[i+16],primer1s_accepted[1],primer2s_accepted[1])
    outputfile.write(">AM_96set_" + str(i+5) + "\n" + targetseq + "\n")


In [167]:
outputfile.close()

In [94]:
# remove ATAG
overhangs2_subset = ['AGTG', 'CAGG', 'ACTC', 'AAAA', 'AGAC', 'CGAA', 'AACC', 
              'TACA', 'TAGA', 'ATGC', 'GATA', 'CTCC', 'GTAA', 'CTGA', 'ACAA', 'AGGA', 'ATTA', 'ACCG', 'GCGA']

In [113]:

# create a mini-U6 for each target in both 18 and 17 basepair guides
def create_u6(target,primerfrag,length,re1,re2):
    ret = primerfrag + bsmbi_sequence + "T" + re1 + minimal_u6 + target[(20 - length):20] + guide_scaffold + re2 + "A" + bsmbi_rc_sequence + "AG"
    return(ret)

#def create_u6_rev(primerfwd,primerrev,re2):
#    ret = primerfwd + bsmbi_sequence + "T" + scaffold_fragment2_with_overlap + re2 + "A" + bsmbi_rc_sequence + revcomp(primerrev)
#    pad = random_sequence(250 - len(ret))
#    return(ret + pad)
    

In [115]:
print(len(create_u6(targets[0],"CCACAGGATATGGC",17,overhangs[0],overhangs[1])))
# print(len(create_u6_rev(primer1s_accepted[2],primer2s_accepted[2],overhangs[1])))

250


In [117]:
outputfile = open("/Users/aaronmck/Desktop/u6_set.fasta","w")
for i in range(0,8):
    u6fwd = create_u6(targets[0],"GGGCTAACCTGAG",18,overhangs[i],overhangs[i+1])
    outputfile.write(">U6fwd_18_" + str(i) + "\n" + u6fwd + "\n")
for i in range(0,8):
    u6fwd = create_u6(targets[0],"CCACAGGATATGGC",17,overhangs[i],overhangs[i+1])
    outputfile.write(">U6fwd_17_" + str(i) + "\n" + u6fwd + "\n")
outputfile.close()

In [119]:
### cas12a section
dimer_pairs = [
'GAATTCCTACTCTCGTAGGTGCCGGTACTCTCCAACCGTTTAATTTCTACTCTTGTAGATATCGAAATCCGCTTGGTAAC',
'GAATTCCTACTCTCGTAGGTGACGCACTCGATAACCGGGATAATTTCTACTCTTGTAGATCGAGCGCCACAGCCCAACTT',
'GAATTCCTACTCTCGTAGGTAGACACGCGATCGGGACCACTAATTTCTACTCTTGTAGATGGACCGCCGTAAGCGAGTAT',
'GAATTCCTACTCTCGTAGGTGGCGTCCAAGTTGCCGTCAATAATTTCTACTCTTGTAGATGCTGCCGGCGGAATGCTATT',
'GAATTCCTACTCTCGTAGGTATGGAAGACGTTTCCGCTAATAATTTCTACTCTTGTAGATATGTCGGGAGCCGCTTTGTA',
'GAATTCCTACTCTCGTAGGTCAGTCTGAGTTCATGCGAGATAATTTCTACTCTTGTAGATATGAGTGGCCTCTCGAATCA',
'GAATTCCTACTCTCGTAGGTGAGTTATCCGACACATCAAATAATTTCTACTCTTGTAGATATGAGATGGATCGCATACTA',
'GAATTCCTACTCTCGTAGGTTAGTCACGGACGAACGATAATAATTTCTACTCTTGTAGATGAGTCGGAGGCGAGTACTTA',
'GAATTCCTACTCTCGTAGGTGTGCCAATGTTAACCGGTCATAATTTCTACTCTTGTAGATTTGTTCATCATAACACTGAA',
'GAATTCCTACTCTCGTAGGTGTGCAATTCCAATACGGCTATAATTTCTACTCTTGTAGATATGACGCCGTGATTATATCA',
'GAATTCCTACTCTCGTAGGTATGGCGGTAACGGTATCCAATAATTTCTACTCTTGTAGATGTGCATGCAGTGTCGGACTA',
'GAATTCCTACTCTCGTAGGTCAGTGGCCGCAGTTATTCCATAATTTCTACTCTTGTAGATCAGCTAACGGTCGCCTAATA',
'GAATTCCTACTCTCGTAGGTTTGAGGAGGCATGTACCCGATAATTTCTACTCTTGTAGATGCCGGTACTCTCCAACCGTT'
]

In [127]:
def create_drs(dimerpair,primerfwd,primerrev,re1,re2):
    return(primerfwd + bsmbi_sequence + "T" + re1 + minimal_u6 + dimerpair + "TTTTTTG" + re2 + "A" + bsmbi_rc_sequence + revcomp(primerrev))
create_drs(dimer_pairs[0],primer1s_accepted[4],primer2s_accepted[4],overhangs[0],overhangs[1])


'GACAACCACCTGAGTCGTCTCTTGCCGAGGGCCTATTTCCCATGATTCCTTCATATTTGCATATACGATAGCTTACCGTAACTTGAAAGTATTTCGATTTCTTGGCTTTATATATCTTGTGGAAAGGACGAAACACCGGAATTCCTACTCTCGTAGGTGCCGGTACTCTCCAACCGTTTAATTTCTACTCTTGTAGATATCGAAATCCGCTTGGTAACTTTTTTGGCAAAGAGACGGCCATATGGTCTCC'

In [130]:
outputfile = open("/Users/aaronmck/Desktop/dr_set.fasta","w")
for i in range(0,len(dimer_pairs)):
    outputfile.write(">DR1_" + str(i) + "\n" + create_drs(dimer_pairs[i],primer1s_accepted[4],primer2s_accepted[4],overhangs[i],overhangs[i+1]) + "\n")
    outputfile.write(">DR2_" + str(i) + "\n" + create_drs(dimer_pairs[i],primer1s_accepted[4],primer2s_accepted[4],overhangs[i],overhangs[i+1]) + "\n")
    outputfile.write(">DR3_" + str(i) + "\n" + create_drs(dimer_pairs[i],primer1s_accepted[4],primer2s_accepted[4],overhangs[i],overhangs[i+1]) + "\n")
    outputfile.write(">DR4_" + str(i) + "\n" + create_drs(dimer_pairs[i],primer1s_accepted[4],primer2s_accepted[4],overhangs[i],overhangs[i+1]) + "\n")
outputfile.close()

In [150]:
## fix maryam's data
from pathlib import Path
bases = set(['A','C','G','T'])

outputfile = open("/Users/aaronmck/Desktop/2021_03_18_oligos_to_order.txt2","w")
inputfile = Path("/Users/aaronmck/Desktop/2021_03_18_oligos_to_order.txt").read_text().splitlines()

for i in range(0,len(inputfile),2):
    guide = inputfile[i].split("_")
    # print(guide[1] + " " + guide[5])
    if guide[5] == '3':
        result = inputfile[i+1].find(guide[1])
        # print(inputfile[i+1][result-1])
        others = bases - set([inputfile[i+1][result-1]])
        for base in others:
            newstr = inputfile[i+1][0:result-1] + base + inputfile[i+1][result:len(inputfile[i+1])]
            outputfile.write(inputfile[i] + "_plus_" + str(base) + "\n" + newstr + "\n")
    else:
        outputfile.write(inputfile[i] + "\n" + inputfile[i+1] + "\n")
        x = 0
outputfile.close()

In [153]:
### make the non-pal library

tracr_old = "gttttagagctaGAAAtagcaagttaaaataaggctagtccgttatcaacttgaaaaagtggcaccgagtcggtgcTTTTTTGAGTCCAGAGA"
tracr_new = "GTTTAAGAGCTATGCTGGAAACAGCATAGCAAGTTTAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGCACCGAGTCGGTGCTTTTTTG"
fwd_bsmbi = "CGTCTCtcaccg"
rvs_bsmbi = "gtttggagacg"
spacer_front = "ATCGA"
spacer_rear = "TCGAT"
print(len(tracr_old + fwd_bsmbi + rvs_bsmbi + spacer_front + spacer_rear))




126


In [159]:
primer_for_libs_fwd = [screen_gc(0.45,0.55,25)[1] for x in range(8)]
primer_for_libs_rev = [screen_gc(0.50,0.60,25)[1] for x in range(8)]

In [161]:
spacer = screen_gc(0.50,0.60,31)[1]

In [168]:
final_250 = open("/Users/aaronmck/Desktop/final_250.fasta.ots.scored.txt")
final_250_to_order = open("/Users/aaronmck/Desktop/final_250.fasta.ots.scored.txt.order","w")

header = final_250.readline()
for line in final_250:
    sp = line.split("\t")
    final_250_to_order.write(">" + sp[0] + "o20\n" + primer_for_libs_fwd[0] + fwd_bsmbi + sp[3][0:20] + tracr_old + spacer + spacer_front + sp[3] + spacer_rear + rvs_bsmbi + primer_for_libs_rev[0] + "\n")
    final_250_to_order.write(">" + sp[0] + "o18\n" + primer_for_libs_fwd[1] + fwd_bsmbi + sp[3][2:20] + tracr_old + spacer + "TA" + spacer_front + sp[3] + spacer_rear + rvs_bsmbi + primer_for_libs_rev[1] + "\n")
    final_250_to_order.write(">" + sp[0] + "o17\n" + primer_for_libs_fwd[2] + fwd_bsmbi + sp[3][3:20] + tracr_old + spacer + "GCA" + spacer_front + sp[3] + spacer_rear + rvs_bsmbi + primer_for_libs_rev[2] + "\n")
    final_250_to_order.write(">" + sp[0] + "n20\n" + primer_for_libs_fwd[3] + fwd_bsmbi + sp[3][0:20] + tracr_new + spacer + spacer_front + sp[3] + spacer_rear + rvs_bsmbi + primer_for_libs_rev[3] + "\n")
    final_250_to_order.write(">" + sp[0] + "n18\n" + primer_for_libs_fwd[4] + fwd_bsmbi + sp[3][2:20] + tracr_new + spacer + "GC" + spacer_front + sp[3] + spacer_rear + rvs_bsmbi + primer_for_libs_rev[4] + "\n")
    final_250_to_order.write(">" + sp[0] + "n17\n" + primer_for_libs_fwd[5] + fwd_bsmbi + sp[3][3:20] + tracr_new + spacer + "ATC" + spacer_front + sp[3] + spacer_rear + rvs_bsmbi + primer_for_libs_rev[5] + "\n")
final_250_to_order.close()